# Gold — Final Reconciliation Statistics tables

Builds two final reconciliation tables the brief specifies:

- `gold.fact_reconciliation_break` — one row per individual mismatch
- `gold.reconciliation_daily` — per (business_date, operator_code)
  aggregate: counts, totals, variance, break category breakdown,
  late_arrival count. This is what Finance opens each morning.

**Inputs**: `gold.matched_pairs`, `gold.breaks_raw`, `silver.partner_events`, `silver.internal_events`
**Outputs**: `gold.fact_reconciliation_break`, `gold.reconciliation_daily`


## Parameters

In [10]:
# Silver normalized tables
silver_partner_table = "silver_partner_events"
silver_internal_table = "silver_internal_events"
# Gold Reconciliation Rules
matched_table = "gold_matched_pairs"
breaks_raw_table = "gold_breaks_raw"

# Final tables for finance
fact_break_table = "fact_reconciliation_break"
daily_table = "reconciliation_daily"


StatementMeta(, d2656c0d-1fb0-434d-8541-460c98e6aad0, 12, Finished, Available, Finished, False)

In [17]:
from pyspark.sql import DataFrame, functions as F, types as T
from functools import reduce
import operator

# Read Silver tables
partner_evt  = spark.table(silver_partner_table)
internal_evt = spark.table(silver_internal_table)

# Read gold rules results
matched      = spark.table(matched_table)
breaks_raw   = spark.table(breaks_raw_table)


StatementMeta(, d2656c0d-1fb0-434d-8541-460c98e6aad0, 19, Finished, Available, Finished, False)

matched_pairs : 11 rows
breaks_raw    : 5 rows


## fact_reconciliation_break

One row per mismatch. Write break_raw table from rule matching engine

The surrogate is a hash of the identifying columns


In [18]:
fact_break = (breaks_raw
    .withColumn("break_id",
        F.xxhash64(F.concat_ws("||",
            F.col("operator_code"),
            F.coalesce(F.col("partner_txn_id"), F.lit("")),
            F.coalesce(F.col("internal_id"),    F.lit("")),
            F.col("break_category"),
            F.col("business_date").cast("string")
        )))
    .select(
        "break_id",
        "operator_code",
        "partner_txn_id",
        "internal_id",
        "break_category",
        "business_date",
        "ingestion_date",
        "partner_amount",
        "internal_amount",
        "variance",
        "source_rule",
    ))

(fact_break.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("business_date", "operator_code")
    .format("delta")
    .saveAsTable(fact_break_table))

display(fact_break.groupBy("break_category").count())


StatementMeta(, d2656c0d-1fb0-434d-8541-460c98e6aad0, 20, Finished, Available, Finished, False)

Wrote fact_reconciliation_break (5 rows)


SynapseWidget(Synapse.DataFrame, 15fd3c55-5ccf-439c-8960-5d241a321ac1)

## reconciliation_daily

The aggregate Finance reads. One row per (business_date, operator_code).

We build this as four CTEs joined together:

1. **per-day partner totals** — count and sum from silver_partner_events
2. **per-day internal totals** — count and sum from silver_internal_events
3. **per-day matched counts** — from gold_matched_pairs (incl. late_arrival)
4. **per-day break category counts** — from fact_reconciliation_break

Then a full outer join across all four on (business_date, operator_code)
because not every day-operator combo has rows in every input.


In [5]:
p_agg = (partner_evt.groupBy("business_date", "operator_code")
    .agg(
        F.count("*").alias("partner_txn_count"),
        F.sum("amount").alias("partner_amount_total"),
    ))

i_agg = (internal_evt.groupBy("business_date", "operator_code")
    .agg(
        F.count("*").alias("internal_txn_count"),
        F.sum("amount").alias("internal_amount_total"),
    ))

# Note: matched_pairs uses business_date_partner
m_agg = (matched
    .withColumnRenamed("business_date_partner", "business_date")
    .groupBy("business_date", "operator_code")
    .agg(
        F.count("*").alias("matched_count"),
        F.sum(F.when(F.col("is_late_arrival"), 1).otherwise(0))
            .alias("late_arrival_count"),
    ))

b_agg = (fact_break.groupBy("business_date", "operator_code")
    .pivot("break_category",
           ["amount_mismatch", "missing_on_platform",
            "missing_at_partner", "orphan_churn"])
    .count()
    .fillna(0))

for cat in ["amount_mismatch", "missing_on_platform",
            "missing_at_partner", "orphan_churn"]:
    if cat in b_agg.columns:
        b_agg = b_agg.withColumnRenamed(cat, f"break_{cat}_count")
    else:
        b_agg = b_agg.withColumn(f"break_{cat}_count", F.lit(0))


break_cat_cols = [F.col(f"break_{c}_count") for c in
                  ["amount_mismatch", "missing_on_platform",
                   "missing_at_partner", "orphan_churn"]]
b_agg = b_agg.withColumn("break_count", reduce(operator.add, break_cat_cols))


StatementMeta(, d2656c0d-1fb0-434d-8541-460c98e6aad0, 7, Finished, Available, Finished, False)

In [19]:
daily = (p_agg
    .join(i_agg, ["business_date", "operator_code"], "full_outer")
    .join(m_agg, ["business_date", "operator_code"], "full_outer")
    .join(b_agg, ["business_date", "operator_code"], "full_outer"))

# Filling NULLS with 0
for col, default in [
    ("partner_txn_count", 0), ("internal_txn_count", 0),
    ("matched_count", 0), ("break_count", 0),
    ("late_arrival_count", 0),
    ("break_amount_mismatch_count", 0),
    ("break_missing_on_platform_count", 0),
    ("break_missing_at_partner_count", 0),
    ("break_orphan_churn_count", 0),
]:
    if col in daily.columns:
        daily = daily.fillna({col: default})

# Variance = partner total - internal total
daily = daily.withColumn("variance",
    F.coalesce(F.col("partner_amount_total"), F.lit(0.0))
    - F.coalesce(F.col("internal_amount_total"), F.lit(0.0)))

final_cols = [
    "business_date", "operator_code",
    "partner_txn_count", "internal_txn_count",
    "matched_count", "break_count",
    "partner_amount_total", "internal_amount_total", "variance",
    "break_amount_mismatch_count", "break_missing_on_platform_count",
    "break_missing_at_partner_count", "break_orphan_churn_count",
    "late_arrival_count",
]
daily = daily.select(*[c for c in final_cols if c in daily.columns])

(daily.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("business_date")
    .format("delta")
    .saveAsTable(daily_table))

display(daily.orderBy("business_date", "operator_code"))


StatementMeta(, d2656c0d-1fb0-434d-8541-460c98e6aad0, 21, Finished, Available, Finished, True)

Wrote reconciliation_daily (3 rows)


SynapseWidget(Synapse.DataFrame, 96300a51-eef7-4185-a866-7713668f3d76)